# EmpathAI Fine-tune (SFT + DPO)
Pipeline: Unsloth QLoRA 4-bit → SFT → DPO → Push HuggingFace

In [1]:
# 1. INSTALL DEPENDENCIES (Bản MỚI NHẤT để fix lỗi Tokenizer)
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl>=0.12.0" "transformers>=4.46.0" peft accelerate bitsandbytes
!pip install fastapi uvicorn datasets huggingface_hub python-dotenv

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-falc585e/unsloth_a6440a5ffc1f476eb842a6d5ab517783
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-falc585e/unsloth_a6440a5ffc1f476eb842a6d5ab517783
  Resolved https://github.com/unslothai/unsloth.git to commit 21e9a91a5746e1146201c8493ae3245fef6c762b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 83.9 MB/s eta 0:00:00
 

In [2]:
# 2. LOGIN HUGGINGFACE
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Đã đăng nhập HuggingFace thành công qua Kaggle Secrets!")
except Exception as e:
    print(f"Lỗi: {e}")
    print("Bạn chưa cài HF_TOKEN trong Kaggle Secrets.")
    print("Vào Settings > Secrets > Add New Secret, tên là HF_TOKEN.")
    # Fallback: điền token thủ công (ĐỪNG SHARE NOTEBOOK)
    # login(token="hf_...")

Đã đăng nhập HuggingFace thành công qua Kaggle Secrets!


In [ ]:
# 3. LOAD MODEL & LORA ADAPTERS
from unsloth import FastLanguageModel
import torch

print("GPU đang dùng:", torch.cuda.get_device_name(0))
print("VRAM trống:", round(torch.cuda.mem_get_info()[0] / 1e9, 2), "GB")

max_seq_length = 2048
dtype = None
load_in_4bit = True

print("\nĐang tải model Llama 3.1 8B (4-bit)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("\nGắn LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                           # tăng từ 16→32: capacity cao hơn, chỉ +~100MB VRAM
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,                  # convention: alpha = r (unsloth recommend) hoặc 2*r
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU đang dùng: Tesla T4
VRAM trống: 15.47 GB

Đang tải model Llama 3.1 8B (4-bit)...
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.



Gắn LoRA adapters...


Unsloth 2026.4.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [ ]:
# 4. LOAD SFT DATASET
from datasets import load_dataset

dataset_id = "thanhhoangnvbg/empathAI-dpo-vi"
print(f"Đang tải dataset từ: {dataset_id}")

sft_train = load_dataset(dataset_id, data_files="sft_train.jsonl", split="train")
sft_dev   = load_dataset(dataset_id, data_files="sft_dev.jsonl",   split="train")

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

sft_train = sft_train.map(format_chat, batched=False)
sft_dev   = sft_dev.map(format_chat, batched=False)

print(f"SFT Train: {len(sft_train)} mẫu | Dev: {len(sft_dev)} mẫu")
print("\nMẫu đầu tiên (500 ký tự đầu):")
print(sft_train[0]['text'][:500])

Đang tải dataset từ: thanhhoangnvbg/empathAI-dpo-vi


sft_train.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

sft_val.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3066 [00:00<?, ? examples/s]

Map:   0%|          | 0/341 [00:00<?, ? examples/s]

SFT Train: 3066 mẫu | Val: 341 mẫu

Mẫu đầu tiên (500 ký tự đầu):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Bạn là EmpathAI, trợ lý CSKH chuyên xử lý khách hàng đang bực tức. Bạn thấu cảm thực sự, nhượng bộ thông minh, KHÔNG BAO GIỜ dùng văn mẫu như 'Chúng tôi rất tiếc' hay 'Theo chính sách công ty'. Bạn luôn đề xuất bồi thường cụ thể và kết thúc bằng câu hỏi mở.<|eot_id|><|start_header_id|>user<|end_header_id|>

Shop bán hàng rác, giao thiếu thì còn muốn sống không? Bỏ tù hết lũ


In [ ]:
# 5. HUẤN LUYỆN SFT
from trl import SFTTrainer
from transformers import TrainingArguments, Trainer
from unsloth import is_bfloat16_supported
import transformers
import inspect

sft_sig = inspect.signature(SFTTrainer.__init__)
trainer_kwargs = {}
if "processing_class" in sft_sig.parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

if "tokenizer" in trainer_kwargs and transformers.__version__ >= "4.46.0":
    _old_init = Trainer.__init__
    def _patched_init(self, *args, **kwargs):
        if "tokenizer" in kwargs:
            kwargs["processing_class"] = kwargs.pop("tokenizer")
        _old_init(self, *args, **kwargs)
    Trainer.__init__ = _patched_init

# [KHIÊN SFT] Chống lỗi 'int' object has no attribute 'mean'
if hasattr(SFTTrainer, "compute_loss"):
    _old_sft_compute_loss = SFTTrainer.compute_loss
    def _patched_sft_compute_loss(self, *args, **kwargs):
        kwargs.pop("num_items_in_batch", None)
        return _old_sft_compute_loss(self, *args, **kwargs)
    SFTTrainer.compute_loss = _patched_sft_compute_loss

# NOTE: packing=True → train loss tính trên packed sequences nên luôn cao hơn val loss
# Đây là bình thường — chỉ dùng val loss để đánh giá convergence

sft_trainer = SFTTrainer(
    model=model,
    **trainer_kwargs,
    train_dataset=sft_train,
    eval_dataset=sft_dev,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        num_train_epochs=3,
        per_device_train_batch_size=1,          # giảm 2→1: giảm ~50% peak VRAM
        per_device_eval_batch_size=2,           # eval batch vừa đủ
        gradient_accumulation_steps=8,          # tăng 4→8: giữ effective batch = 1*8 = 8
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=0.3,
        seed=3407,
        output_dir="empathAI-sft",
        report_to="none",
    ),
)

print("Bắt đầu huấn luyện SFT...")
sft_trainer.train()
print("Huấn luyện SFT hoàn tất!")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/3066 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/341 [00:00<?, ? examples/s]

Bắt đầu huấn luyện SFT...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,066 | Num Epochs = 1 | Total steps = 192
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


AttributeError: 'int' object has no attribute 'mean'

In [ ]:
# 5b. ĐÁNH GIÁ SFT TRÊN TEST SET
sft_test = load_dataset(dataset_id, data_files="sft_test.jsonl", split="train")
sft_test = sft_test.map(format_chat, batched=False)
print(f"SFT Test: {len(sft_test)} mẫu")

# SFTTrainer.evaluate() không tự tokenize dataset mới → phải tokenize thủ công
def tokenize_sft(example):
    tokenized = tokenizer(example["text"], truncation=True, max_length=2048, padding=False)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

sft_test_tok = sft_test.map(tokenize_sft, remove_columns=sft_test.column_names)

sft_test_results = sft_trainer.evaluate(eval_dataset=sft_test_tok)
print(f"SFT Test Loss: {sft_test_results['eval_loss']:.4f}")

In [ ]:
# 6. LOAD DPO DATASET
dpo_train = load_dataset(dataset_id, data_files="dpo_train.jsonl", split="train")
dpo_dev   = load_dataset(dataset_id, data_files="dpo_dev.jsonl",   split="train")

# Kiểm tra cấu trúc dữ liệu trước
print("Cấu trúc 1 mẫu DPO:", {k: type(v).__name__ for k, v in dpo_train[0].items()})

def format_dpo(example):
    prompt   = list(example["prompt"])
    chosen   = list(example["chosen"])
    rejected = list(example["rejected"])
    return {
        "prompt": tokenizer.apply_chat_template(
            prompt, tokenize=False, add_generation_prompt=True
        ),
        "chosen": tokenizer.apply_chat_template(
            prompt + chosen, tokenize=False, add_generation_prompt=False
        ),
        "rejected": tokenizer.apply_chat_template(
            prompt + rejected, tokenize=False, add_generation_prompt=False
        ),
    }

dpo_train = dpo_train.map(format_dpo)
dpo_dev   = dpo_dev.map(format_dpo)
print(f"DPO Train: {len(dpo_train)} mẫu | Dev: {len(dpo_dev)} mẫu")

In [ ]:
# 7. HUẤN LUYỆN DPO
from trl import DPOTrainer
from transformers import TrainingArguments, Trainer
import inspect
from unsloth import is_bfloat16_supported

# ------------------------------------------------------------------
# KHÔI PHỤC LÕI GỐC (Chặn việc dán đè code nhiều lần làm hỏng RAM)
if "_dpo_get_batch" in globals():
    DPOTrainer.get_batch_samples = _dpo_get_batch
if "_old_compute_loss" in globals():
    DPOTrainer.compute_loss = _old_compute_loss
if "_old_log" in globals():
    DPOTrainer.log = _old_log
# ------------------------------------------------------------------

# ==============================================================================
# 🛡️ 3 LÁ CHẮN BẢO VỆ TUYỆT ĐỐI CHO KAGGLE 🛡️
dpo_sig = inspect.signature(DPOTrainer.__init__)
dpo_kwargs = {}
if "processing_class" in dpo_sig.parameters:
    dpo_kwargs["processing_class"] = tokenizer
else:
    dpo_kwargs["tokenizer"] = tokenizer

# [KHIÊN 1] Chống Transformers gọi nhầm get_batch_samples
if hasattr(DPOTrainer, "get_batch_samples") and hasattr(Trainer, "get_batch_samples"):
    _dpo_get_batch = DPOTrainer.get_batch_samples
    def _patched_get_batch(self, *args, **kwargs):
        if len(args) > 1 and isinstance(args[1], int):
            return Trainer.get_batch_samples(self, *args, **kwargs)
        return _dpo_get_batch(self, *args, **kwargs)
    DPOTrainer.get_batch_samples = _patched_get_batch

# [KHIÊN 2] Cắt bỏ rác 'num_items_in_batch' do Transformers ném vào compute_loss
if hasattr(DPOTrainer, "compute_loss"):
    _old_compute_loss = DPOTrainer.compute_loss
    def _patched_compute_loss(self, *args, **kwargs):
        kwargs.pop("num_items_in_batch", None)
        return _old_compute_loss(self, *args, **kwargs)
    DPOTrainer.compute_loss = _patched_compute_loss

# [KHIÊN 3] Cắt bỏ rác 'start_time' do Transformers ném vào log
if hasattr(DPOTrainer, "log"):
    _old_log = DPOTrainer.log
    def _patched_log(self, logs, *args, **kwargs):
        return _old_log(self, logs)
    DPOTrainer.log = _patched_log
# ==============================================================================

# NOTE: DPO xử lý đồng thời chosen + rejected → memory = 2x SFT
# max_length=1024 thay vì 2048 → giảm ~50% VRAM, đủ cho data CSKH tiếng Việt

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    beta=0.1,
    max_length=1024,                # giảm 2048→1024: DPO dùng 2x memory (chosen+rejected)
    max_prompt_length=512,          # giảm 1536→512: prompt CSKH thường ~200-300 tokens
    train_dataset=dpo_train,
    eval_dataset=dpo_dev,
    **dpo_kwargs,
    args=TrainingArguments(
        num_train_epochs=2,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,           # giảm 2→1: eval DPO rất nặng memory
        gradient_accumulation_steps=4,          # effective batch = 1*4 = 4
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        output_dir="empathAI-dpo",
        report_to="none",
    ),
)

print("Bắt đầu huấn luyện DPO...")
dpo_trainer.train()
print("Huấn luyện DPO hoàn tất!")

In [ ]:
# 7b. ĐÁNH GIÁ DPO TRÊN TEST SET
dpo_test = load_dataset(dataset_id, data_files="dpo_test.jsonl", split="train")
dpo_test = dpo_test.map(format_dpo)
print(f"DPO Test: {len(dpo_test)} mẫu")

dpo_test_results = dpo_trainer.evaluate(eval_dataset=dpo_test)
print(f"DPO Test Loss: {dpo_test_results['eval_loss']:.4f}")

In [ ]:
# 8. PUSH MODEL LÊN HUGGINGFACE
hf_repo_name = "thanhhoangnvbg/empathAI-llama3.1-8b"

print(f"Đang push LoRA adapter lên {hf_repo_name}...")
print("(save_method=lora: nhanh ~2-3 phút, cần load kèm base model khi inference)")
try:
    model.push_to_hub_merged(hf_repo_name, tokenizer, save_method="lora")
    print(f"\nHoàn tất! Model đã có tại: https://huggingface.co/{hf_repo_name}")
except Exception as e:
    print(f"Lỗi: {e}")
    print("Kiểm tra lại HF_TOKEN và quyền write vào repo.")

In [ ]:
# 9. TEST MODEL (OPTIONAL)
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = "Bạn là EmpathAI, trợ lý CSKH chuyên xử lý khách hàng đang bực tức. Bạn thấu cảm thực sự, nhượng bộ thông minh, KHÔNG BAO GIỜ dùng văn mẫu."
test_input = "Tôi thực sự nổi điên với shop của bạn. Mua hàng từ tuần trước mà giờ vẫn báo Đang chờ lấy hàng, trong khi shop chat thì xin lỗi suông. Dẹp hệ thống của các người đi!"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": test_input}
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)
response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("=" * 60)
print("KHÁCH HÀNG:")
print(test_input)
print("\nEMPATHAI:")
print(response)
print("=" * 60)